In [ ]:
from google.colab import files
files.upload()  # select kaggle.json when prompted

import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

!pip install -q kaggle
!kaggle datasets download -d apoorvaappz/global-super-store-dataset
!unzip -o global-super-store-dataset.zip

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/apoorvaappz/global-super-store-dataset
License(s): DbCL-1.0
100% 11.1M/11.1M [00:00<00:00, 67.7MB/s]

Archive:  global-super-store-dataset.zip
  inflating: Global_Superstore2.csv  
  inflating: Global_Superstore2.xlsx  


In [ ]:
import pandas as pd

df = pd.read_csv('Global_Superstore2.csv', encoding='latin1')
print(df.shape)
print(df.columns.tolist())
print(df.isnull().sum())
print(df.info())

(51290, 24)
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'City', 'State', 'Country', 'Postal Code', 'Market', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit', 'Shipping Cost', 'Order Priority']
Row ID                0
Order ID              0
Order Date            0
Ship Date             0
Ship Mode             0
Customer ID           0
Customer Name         0
Segment               0
City                  0
State                 0
Country               0
Postal Code       41296
Market                0
Region                0
Product ID            0
Category              0
Sub-Category          0
Product Name          0
Sales                 0
Quantity              0
Discount              0
Profit                0
Shipping Cost         0
Order Priority        0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51290 entries, 0 to 51289
Data columns (tota

In [ ]:
# Drop unneeded columns
df = df.drop(columns=['Postal Code'], errors='ignore')
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

# Fix dates
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True, errors='coerce')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], dayfirst=True, errors='coerce')

# Drop duplicates
df = df.drop_duplicates()

# Shipping delay
df['Shipping Delay Days'] = (df['Ship Date'] - df['Order Date']).dt.days

# Time features
df['Order Year'] = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.to_period('M').astype(str)

In [ ]:
# Profit margin
df['Profit Margin %'] = (df['Profit'] / df['Sales']) * 100

# Discount bucket — for discount-impact analysis
df['Discount Bucket'] = pd.cut(
    df['Discount'],
    bins=[-0.01, 0, 0.2, 0.4, 1],
    labels=['No Discount', 'Low (0-20%)', 'Medium (20-40%)', 'High (40%+)']
)

# Net profit after shipping
df['Net Profit After Shipping'] = df['Profit'] - df['Shipping Cost']

In [ ]:
import datetime as dt

snapshot_date = df['Order Date'].max() + dt.timedelta(days=1)

rfm = df.groupby('Customer ID').agg({
    'Order Date': lambda x: (snapshot_date - x.max()).days,
    'Order ID': 'nunique',
    'Sales': 'sum'
}).reset_index()
rfm.columns = ['Customer ID', 'Recency', 'Frequency', 'Monetary']

# Simple segment labels
rfm['Segment'] = pd.qcut(rfm['Monetary'], 4, labels=['Low', 'Mid-Low', 'Mid-High', 'High'])
rfm.to_csv('rfm_output.csv', index=False)

In [ ]:
df.to_csv('cleaned_global_superstore.csv', index=False)

from google.colab import files
files.download('cleaned_global_superstore.csv')
files.download('rfm_output.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>